# GENE QUEST 環境確認ノートブック

**目的**：この実行環境（Kaggle Notebook またはローカル）で、音声合成と動画合成に必要なものが揃っているかを確認する。

- 必要なもの：Python 3.10 以上、FFmpeg、（あれば）GPU と CUDA 版 PyTorch
- 入力：なし ／ 出力：画面の表示のみ
- **GPU が使えない場合**：CPU でも全工程を実行できる（音声合成は CPU で実時間の約0.3倍の処理時間。246行で約8分）
- **Kaggle の設定**：右側の Settings で Accelerator を「GPU T4 x2」または「GPU P100」、Internet を On にする（モデルのダウンロードに必要）。
  GPU の利用枠（週あたりの時間）には上限がある。上限を超える使い方や、セッション制限の回避はしないこと。
- **エラー時**：`nvidia-smi` が失敗する → Accelerator が None。`ffmpeg` がない → Kaggle には通常入っている。ローカルでは OS のパッケージで入れる

In [ ]:
import importlib.util


def ensure(pkgs: dict) -> None:
    """入っていないパッケージだけを入れる（pip がなければ uv を使う）。フル版の unidic は MeCab の初期化に失敗するので外す。"""
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    has_pip = importlib.util.find_spec("pip") is not None
    if missing:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] if has_pip else ["uv", "pip", "install", "--python", sys.executable]
        subprocess.run(cmd + missing, check=True)
    if importlib.util.find_spec("unidic") is not None:
        cmd = [sys.executable, "-m", "pip", "uninstall", "-y", "-q"] if has_pip else ["uv", "pip", "uninstall", "--python", sys.executable]
        subprocess.run(cmd + ["unidic"], check=True)
    print("追加したパッケージ:", missing or "なし")

import os, sys, subprocess, shutil, platform
from pathlib import Path
ON_KAGGLE = Path("/kaggle").exists()
# プロジェクトの場所：Kaggle では Dataset として追加した production/gene-quest を /kaggle/working にコピーして使う
PROJECT = Path(os.environ.get("GQ_PROJECT", "/kaggle/working/gene-quest" if ON_KAGGLE else Path.cwd().parent))
SRC = Path("/kaggle/input/gene-quest") if ON_KAGGLE else None
if ON_KAGGLE and not PROJECT.exists() and SRC and SRC.exists():
    shutil.copytree(SRC, PROJECT)
print("project:", PROJECT, "exists:", PROJECT.exists())

In [ ]:
print("Python", sys.version.split()[0], platform.platform())
print("ffmpeg:", shutil.which("ffmpeg"))
gpu = ""
if shutil.which("nvidia-smi"):  # GPU のない環境には nvidia-smi 自体がない
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "なし（CPU で実行）")
try:
    import torch
    print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
except ImportError:
    print("torch 未インストール")
du = shutil.disk_usage(PROJECT if PROJECT.exists() else "/")
print(f"空き容量 {du.free/1e9:.1f} GB")